# import libraries

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(34):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-4b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

transcoder_all/layer_0_width_16k_l0_smal(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_1_width_16k_l0_smal(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_2_width_16k_l0_smal(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_3_width_16k_l0_smal(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_4_width_16k_l0_smal(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

...

transcoder_all/layer_29_width_16k_l0_sma(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_30_width_16k_l0_sma(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_31_width_16k_l0_sma(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_32_width_16k_l0_sma(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

transcoder_all/layer_33_width_16k_l0_sma(…):   0%|          | 0.00/336M [00:00<?, ?B/s]

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-4b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-4b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-3-4b-pt into HookedTransformer


# 1. get top influential features

In [6]:
import json
import glob
from collections import defaultdict

In [7]:
GRAPH_DIR = "./graphs/gemma-3-4b"  # ← set this
DESCRIBED_LAYERS = {9, 17, 22, 29}

In [8]:
def parse_local_feat(node):
    """Extract (layer_int, local_feat_idx) from jsNodeId."""
    js = node.get("jsNodeId", "")
    try:
        layer_feat, _ = js.rsplit("-", 1)
        layer_str, feat_str = layer_feat.split("_", 1)
        return int(layer_str), int(feat_str)
    except Exception:
        return None, None

In [9]:
all_features = defaultdict(float)
 
for fpath in sorted(glob.glob(f"{GRAPH_DIR}/step-*.json")):
    with open(fpath) as f:
        data = json.load(f)
    for node in data.get("nodes", []):
        if "transcoder" not in node.get("feature_type", ""):
            continue
        layer, local_feat = parse_local_feat(node)
        if layer is None:
            continue
        inf = abs(node.get("influence", 0) or 0)
        all_features[(layer, local_feat)] += inf

### top ten from all layers

In [10]:
top_10_overall = sorted(all_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 OVERALL (all layers)")
print("=" * 70)
for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}  {in_desc}")

TOP 10 OVERALL (all layers)
 1. L29 F 1784  influence=  7.1921  ✓ DESCRIBED
 2. L32 F  156  influence=  7.1858    undescribed
 3. L29 F 2081  influence=  7.1153  ✓ DESCRIBED
 4. L29 F  107  influence=  7.0198  ✓ DESCRIBED
 5. L31 F 1432  influence=  6.9249    undescribed
 6. L28 F 3512  influence=  6.8427    undescribed
 7. L26 F 3121  influence=  6.7691    undescribed
 8. L31 F 5799  influence=  6.7016    undescribed
 9. L33 F  294  influence=  6.6289    undescribed
10. L33 F10133  influence=  6.5203    undescribed


### top ten from only described layers

In [11]:
described_features = {lf: s for lf, s in all_features.items() if lf[0] in DESCRIBED_LAYERS}
top_10_described_sorted = sorted(described_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)")
print("=" * 70)
if top_10_described_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_described_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_described = [lf for lf, _ in top_10_described_sorted]

TOP 10 FROM DESCRIBED LAYERS ONLY (9, 17, 22, 29)
 1. L29 F 1784  influence=  7.1921
 2. L29 F 2081  influence=  7.1153
 3. L29 F  107  influence=  7.0198
 4. L29 F   80  influence=  5.9334
 5. L29 F 3347  influence=  5.9326
 6. L29 F 9459  influence=  5.7328
 7. L29 F11982  influence=  5.6974
 8. L29 F 6449  influence=  5.6277
 9. L29 F   33  influence=  5.5697
10. L29 F  440  influence=  5.3364


### top ten from every other layer

In [12]:
other_features = {lf: s for lf, s in all_features.items() if lf[0] not in DESCRIBED_LAYERS}
top_10_undescribed_sorted = sorted(other_features.items(), key=lambda x: -x[1])[:10]
 
print("=" * 70)
print("TOP 10 FROM OTHER LAYERS (not 9, 17, 22, 29)")
print("=" * 70)
if top_10_undescribed_sorted:
    for i, ((layer, feat), score) in enumerate(top_10_undescribed_sorted, 1):
        print(f"{i:2d}. L{layer:2d} F{feat:5d}  influence={score:8.4f}")
else:
    print("(none found)")

top_10_undescribed = [lf for lf, _ in top_10_undescribed_sorted]

TOP 10 FROM OTHER LAYERS (not 9, 17, 22, 29)
 1. L32 F  156  influence=  7.1858
 2. L31 F 1432  influence=  6.9249
 3. L28 F 3512  influence=  6.8427
 4. L26 F 3121  influence=  6.7691
 5. L31 F 5799  influence=  6.7016
 6. L33 F  294  influence=  6.6289
 7. L33 F10133  influence=  6.5203
 8. L33 F 1372  influence=  6.4285
 9. L33 F  316  influence=  6.3146
10. L31 F 1174  influence=  6.3012


### summary

In [13]:
print(f"Total unique transcoder features: {len(all_features)}")
print(f"Features in described layers:     {len(described_features)}")
print(f"Features in other layers:         {len(other_features)}")

Total unique transcoder features: 2401
Features in described layers:     323
Features in other layers:         2078


# get clerp description for described layers (top ten only)

In [14]:
import requests
import time

def fetch_clerps_batch(features_list, model_id="gemma-3-4b", slug_suffix="gemmascope-2-res-16k", delay=0.3):
    """Fetch clerp descriptions for a list of (layer, feature_idx) tuples."""
    clerps = {}
    
    for i, (layer, feat) in enumerate(features_list):
        source = f"{layer}-{slug_suffix}"
        url = f"https://www.neuronpedia.org/api/feature/{model_id}/{source}/{feat}"
        
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                exps = r.json().get("explanations", [])
                desc = exps[0].get("description") if exps else None
                clerps[(layer, feat)] = desc
                status = "✓" if desc else "✗ (no desc)"
            else:
                clerps[(layer, feat)] = None
                status = f"✗ (HTTP {r.status_code})"
        except Exception as e:
            clerps[(layer, feat)] = None
            status = f"✗ ({str(e)[:30]})"
        
        print(f"  [{i+1}/{len(features_list)}] L{layer:2d} F{feat:5d}: {status}")
        time.sleep(delay)
    
    return clerps

# Extract the top 10 described features
features_to_fetch = [lf for lf, _ in top_10_described_sorted]

# Fetch clerps
print("Fetching clerps for top 10 described features...")
clerps = fetch_clerps_batch(features_to_fetch)

# Print results
print("\n" + "="*70)
print("TOP 10 DESCRIBED FEATURES WITH CLERPS")
print("="*70)
for (layer, feat), score in top_10_described_sorted:
    desc = clerps.get((layer, feat))
    print(f"L{layer:2d} F{feat:5d} (influence={score:8.4f}): {desc or '[no description found]'}")

Fetching clerps for top 10 described features...
  [1/10] L29 F 1784: ✓
  [2/10] L29 F 2081: ✓
  [3/10] L29 F  107: ✓
  [4/10] L29 F   80: ✓
  [5/10] L29 F 3347: ✓
  [6/10] L29 F 9459: ✓
  [7/10] L29 F11982: ✓
  [8/10] L29 F 6449: ✓
  [9/10] L29 F   33: ✓
  [10/10] L29 F  440: ✓

TOP 10 DESCRIBED FEATURES WITH CLERPS
L29 F 1784 (influence=  7.1921): legal and financial penalties
L29 F 2081 (influence=  7.1153): marine organisms
L29 F  107 (influence=  7.0198): a adjective
L29 F   80 (influence=  5.9334): loneliness and negative emotions
L29 F 3347 (influence=  5.9326): variable assignments and numbers
L29 F 9459 (influence=  5.7328): rectangles
L29 F11982 (influence=  5.6974): `try` blocks for error handling
L29 F 6449 (influence=  5.6277): model refusal messages
L29 F   33 (influence=  5.5697): Italian subjects or cuisine
L29 F  440 (influence=  5.3364): Hebrew and Arabic words


# intervention

### all 20 features suppressed

In [15]:
# Your prompt
prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

# Target position (typically -1 for last token, or specify the rhyme token position)
TARGET_POS = -1

In [16]:
all_features = top_10_described + top_10_undescribed

intervention_tuples = [
    (layer, TARGET_POS, feat, 0.0)
    for layer, feat in all_features
]

print("=" * 70)
print("INTERVENTION TUPLES (all 20 features, suppressed at target position)")
print("=" * 70)
for i, (layer, pos, feat, val) in enumerate(intervention_tuples, 1):
    print(f"{i:2d}. Layer {layer:2d}, Position {pos:3d}, Feature {feat:5d}, Value {val}")

INTERVENTION TUPLES (all 20 features, suppressed at target position)
 1. Layer 29, Position  -1, Feature  1784, Value 0.0
 2. Layer 29, Position  -1, Feature  2081, Value 0.0
 3. Layer 29, Position  -1, Feature   107, Value 0.0
 4. Layer 29, Position  -1, Feature    80, Value 0.0
 5. Layer 29, Position  -1, Feature  3347, Value 0.0
 6. Layer 29, Position  -1, Feature  9459, Value 0.0
 7. Layer 29, Position  -1, Feature 11982, Value 0.0
 8. Layer 29, Position  -1, Feature  6449, Value 0.0
 9. Layer 29, Position  -1, Feature    33, Value 0.0
10. Layer 29, Position  -1, Feature   440, Value 0.0
11. Layer 32, Position  -1, Feature   156, Value 0.0
12. Layer 31, Position  -1, Feature  1432, Value 0.0
13. Layer 28, Position  -1, Feature  3512, Value 0.0
14. Layer 26, Position  -1, Feature  3121, Value 0.0
15. Layer 31, Position  -1, Feature  5799, Value 0.0
16. Layer 33, Position  -1, Feature   294, Value 0.0
17. Layer 33, Position  -1, Feature 10133, Value 0.0
18. Layer 33, Position  -1, Fe

In [17]:
print("\n" + "=" * 70)
print("GENERATING WITH ORIGINAL FEATURES (no intervention)")
print("=" * 70)

pre_intervention_text = model.feature_intervention_generate(
    prompt,
    [],  # empty list = no interventions
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{pre_intervention_text}")

# ─────────────────────────────────────────────────────────────
# Generate with all 20 features suppressed
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("GENERATING WITH ALL 20 FEATURES SUPPRESSED")
print("=" * 70)

post_intervention_text = model.feature_intervention_generate(
    prompt,
    intervention_tuples,
    do_sample=False,
    verbose=True
)[0]

print(f"\nGenerated text:\n{post_intervention_text}")


GENERATING WITH ORIGINAL FEATURES (no intervention)


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
He ate it and then he had to crap it

GENERATING WITH ALL 20 FEATURES SUPPRESSED


  0%|          | 0/10 [00:00<?, ?it/s]


Generated text:
A rhyming couplet:
He saw a carrot and had to grab it,
питы, питы, питы, питы


In [18]:
print("\n" + "=" * 70)
print("COMPARISON: PRE vs POST INTERVENTION")
print("=" * 70)

from circuit_tracer.utils.demo_utils import display_generations_comparison

display_generations_comparison(
    prompt,
    [pre_intervention_text],
    [post_intervention_text]
)


COMPARISON: PRE vs POST INTERVENTION


### suppressed one at a time (top five)

In [19]:
print("Suppressing one feature at a time to isolate effects...")

ablation_results = []

for layer, feat in all_features[:5]:  # just first 5 for speed
    single_intervention = [(layer, TARGET_POS, feat, 0.0)]
    
    try:
        ablated_text = model.feature_intervention_generate(
            prompt,
            single_intervention,
            do_sample=False,
            verbose=False
        )[0]
        
        # Check if output changed
        changed = ablated_text != pre_intervention_text
        status = "CHANGED" if changed else "unchanged"
        
        # Get clerp if available
        clerp_desc = clerps.get((layer, feat)) if (layer, feat) in clerps else None
        clerp_str = f" | {clerp_desc}" if clerp_desc else ""
        
        ablation_results.append({
            'layer': layer,
            'feat': feat,
            'changed': changed,
            'text': ablated_text,
            'clerp': clerp_desc
        })
        
        print(f"\nL{layer:2d} F{feat:5d}: {status}{clerp_str}")
        if changed:
            print(f"  → {ablated_text}")
    
    except Exception as e:
        print(f"L{layer:2d} F{feat:5d}: ERROR ({str(e)[:50]})")

Suppressing one feature at a time to isolate effects...

L29 F 1784: CHANGED | legal and financial penalties
  → A rhyming couplet:
He saw a carrot and had to grab it,
如果你想吃它,你必须把它拔出来

L29 F 2081: CHANGED | marine organisms
  → A rhyming couplet:
He saw a carrot and had to grab it,
sang the song of the rabbit.
He saw

L29 F  107: CHANGED | a adjective
  → A rhyming couplet:
He saw a carrot and had to grab it,
tle, tle, tle, tle, tle,

L29 F   80: CHANGED | loneliness and negative emotions
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Advertiser
He saw a carrot and had to grab

L29 F 3347: CHANGED | variable assignments and numbers
  → A rhyming couplet:
He saw a carrot and had to grab it,
 Той побачив моркву і довелося


# pseudo-clerp

In [23]:
def pseudo_clerp_topk(model, layer, local_feat, tokenizer, top_k=10):
    """
    Project feature decoder direction through unembedding matrix.
    Returns top-k predicted tokens for this feature.
    """
    tc = model.transcoders[layer]
    # W_dec is the direction in residual space the feature 'writes' to
    W_dec = tc.W_dec[local_feat] # shape: [d_model]
    
    with torch.no_grad():
        # model.unembed.W_U shape is usually [d_model, vocab_size]
        W_U = model.unembed.W_U 
        
        # We want to project W_dec (d_model) onto each vocab entry (d_model)
        # So we do: [vocab_size, d_model] @ [d_model]
        # We use W_U.T to get [vocab_size, d_model]
        logits = torch.matmul(W_U.T, W_dec.to(W_U.dtype)) # result: [vocab_size]
    
    # Get top-k tokens
    top_ids = logits.topk(top_k).indices.tolist()
    top_tokens = [tokenizer.decode([i]) for i in top_ids]
    
    return top_tokens

# ─────────────────────────────────────────────────────────────
# Run pseudo-clerp for all top 10 overall features
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES")
print("=" * 70)

for i, ((layer, feat), score) in enumerate(top_10_overall, 1):
    tokens = pseudo_clerp_topk(model, layer, feat, tokenizer, top_k=10)
    
    # Check if this is a described feature
    in_desc = "✓ DESCRIBED" if layer in DESCRIBED_LAYERS else "  undescribed"
    
    print(f"\n{i:2d}. L{layer:2d} F{feat:5d} (influence={score:8.4f}) {in_desc}")
    print(f"    Top tokens: {tokens}")

PSEUDO-CLERP FOR TOP 10 OVERALL FEATURES

 1. L29 F 1784 (influence=  7.1921) ✓ DESCRIBED
    Top tokens: [' Efter', ' För', ' Geb', ' Rie', ' Vä', ' Kas', 'Efter', ' Arbe', 'Kam', ' Stol']

 2. L32 F  156 (influence=  7.1858)   undescribed
    Top tokens: ['<unused338>', 'ꗜ', 'ri', '𒅤', ' PLDNN', '𒉶', 'rik', ' rid', '𒂷', 'MalayMarks']

 3. L29 F 2081 (influence=  7.1153) ✓ DESCRIBED
    Top tokens: ['Gre', '<strong>', '<sub>', ' Gre', ' gre', 'Gr', ' calon', ' Ref', ' ল', '<i>']

 4. L29 F  107 (influence=  7.0198) ✓ DESCRIBED
    Top tokens: ['<unused366>', '<unused687>', '<unused192>', '<unused201>', '<unused1166>', '<unused108>', '<unused1207>', 'Referências', '.</', '<unused712>']

 5. L31 F 1432 (influence=  6.9249)   undescribed
    Top tokens: [' פ', ' การ', ' ק', ' ס', ' ו', ' מ', ' ג', ' ש', ' כ', ' അ']

 6. L28 F 3512 (influence=  6.8427)   undescribed
    Top tokens: [' str', ' Lo', ' oznac', '條', 'が登場', ' Str', ' Card', 'MathML', ' Ministries', ' aggravate']

 7. L26 F 312